# Retail Sales Performance & Profitability Analysis

**Business question:** Which product categories and regions drive sales and profit,
and how can a retail company optimize discounting and shipping strategies to improve profitability?

**Dataset:** Superstore sales dataset (public, 8,399 line items).

This notebook cleans the data, computes profitability KPIs, produces charts and
exports the numbers consumed by the dashboard (`docs/dashboard.json`).

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data" / "superstore.csv"
OUT_CHARTS = ROOT / "outputs" / "charts"
OUT_DOCS = ROOT / "docs" / "data"
OUT_CHARTS.mkdir(parents=True, exist_ok=True)
OUT_DOCS.mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
PALETTE = ["#1f4e79", "#2e75b6", "#9dc3e6", "#c55a11", "#f4b183"]

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head()

## 2. Clean the data
- Parse dates.
- Drop rows without sales or profit.
- Clip discounts to a sane range [0, 1].
- Create calculated fields: profit margin, shipping cost ratio, lead time.

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

before = len(df)
df = df.dropna(subset=["Sales", "Profit", "Order Date"])
df = df[df["Sales"] > 0]
df["Discount"] = df["Discount"].clip(lower=0, upper=1)
df["Shipping Cost"] = df["Shipping Cost"].fillna(0)

df["Profit Margin"] = df["Profit"] / df["Sales"]
df["Shipping Cost Ratio"] = df["Shipping Cost"] / df["Sales"]
df["Lead Time Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.to_period("M").astype(str)

print(f"Dropped {before - len(df)} rows during cleaning. Remaining: {len(df):,}")

## 3. Headline KPIs

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales
avg_discount = df["Discount"].mean()
avg_lead_time = df["Lead Time Days"].mean()
shipping_ratio = df["Shipping Cost"].sum() / total_sales
orders = df["Order ID"].nunique()

print(f"Total sales:        ${total_sales:,.0f}")
print(f"Total profit:       ${total_profit:,.0f}")
print(f"Profit margin:      {profit_margin:.1%}")
print(f"Average discount:   {avg_discount:.1%}")
print(f"Avg lead time:      {avg_lead_time:.1f} days")
print(f"Shipping cost ratio:{shipping_ratio:.1%}")
print(f"Distinct orders:    {orders:,}")

## 4. Sales & profit by product category
Office Supplies dominates volume, but Technology carries the highest margin.

In [ ]:
cat = (
    df.groupby("Product Category")
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"),
           orders=("Order ID", "nunique"))
      .reset_index()
)
cat["profit_margin_pct"] = (cat["total_profit"] / cat["total_sales"] * 100).round(2)
cat["avg_discount_pct"] = (cat["avg_discount"] * 100).round(2)
cat = cat.sort_values("total_sales", ascending=False)
cat[["total_sales", "total_profit"]] = cat[["total_sales", "total_profit"]].round(2)
cat

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(cat))
w = 0.38
ax.bar(x - w / 2, cat["total_sales"], w, label="Sales", color=PALETTE[0])
ax.bar(x + w / 2, cat["total_profit"], w, label="Profit", color=PALETTE[3])
ax.set_xticks(x)
ax.set_xticklabels(cat["Product Category"])
ax.set_ylabel("USD")
ax.set_title("Sales vs Profit by Product Category")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "sales_profit_by_category.png", dpi=120)
plt.close(fig)

## 5. Profit by region
Some regions are profitable but thin; others are loss-making.

In [ ]:
reg = (
    df.groupby("Region")
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"))
      .reset_index()
)
reg["profit_margin_pct"] = (reg["total_profit"] / reg["total_sales"] * 100).round(2)
reg["avg_discount_pct"] = (reg["avg_discount"] * 100).round(2)
reg = reg.sort_values("total_profit", ascending=False)
reg[["total_sales", "total_profit"]] = reg[["total_sales", "total_profit"]].round(2)
reg

In [ ]:
fig, ax = plt.subplots()
colors = [PALETTE[0] if p >= 0 else PALETTE[3] for p in reg["total_profit"]]
ax.barh(reg["Region"], reg["total_profit"], color=colors)
ax.set_xlabel("Total profit (USD)")
ax.set_title("Profit by Region")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "profit_by_region.png", dpi=120)
plt.close(fig)

## 6. Discount impact on profitability
Discounts above ~15% systematically destroy margin.

In [ ]:
bins = [-0.001, 0.0, 0.05, 0.10, 0.15, 0.20, 1.0]
labels = ["0%", "1-5%", "6-10%", "11-15%", "16-20%", "20%+"]
df["Discount Band"] = pd.cut(df["Discount"], bins=bins, labels=labels)

disc = (
    df.groupby("Discount Band", observed=True)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           line_items=("Sales", "size"))
      .reset_index()
)
disc["profit_margin_pct"] = (disc["total_profit"] / disc["total_sales"] * 100).round(2)
disc["total_sales"] = disc["total_sales"].round(2)
disc["total_profit"] = disc["total_profit"].round(2)
disc

In [ ]:
fig, ax = plt.subplots()
ax.bar(disc["Discount Band"].astype(str), disc["profit_margin_pct"],
       color=[PALETTE[0] if m >= 0 else PALETTE[3] for m in disc["profit_margin_pct"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Profit margin (%)")
ax.set_xlabel("Discount band")
ax.set_title("Profit Margin by Discount Band")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "discount_impact.png", dpi=120)
plt.close(fig)

## 7. Shipping efficiency
Delivery Truck is cheapest per order but slower; Regular Air is fastest.

In [ ]:
ship = (
    df.groupby("Ship Mode")
      .agg(line_items=("Sales", "size"),
           total_shipping_cost=("Shipping Cost", "sum"),
           avg_shipping_cost=("Shipping Cost", "mean"),
           avg_lead_time=("Lead Time Days", "mean"),
           total_sales=("Sales", "sum"))
      .reset_index()
)
ship["shipping_cost_pct_of_sales"] = (ship["total_shipping_cost"] / ship["total_sales"] * 100).round(2)
ship["avg_shipping_cost"] = ship["avg_shipping_cost"].round(2)
ship["avg_lead_time"] = ship["avg_lead_time"].round(1)
ship["total_shipping_cost"] = ship["total_shipping_cost"].round(2)
ship = ship.sort_values("avg_shipping_cost")
ship

In [ ]:
fig, ax = plt.subplots()
ax.scatter(ship["avg_lead_time"], ship["avg_shipping_cost"],
           s=ship["line_items"] / 5, color=PALETTE[1], alpha=0.8)
for _, r in ship.iterrows():
    ax.annotate(r["Ship Mode"], (r["avg_lead_time"], r["avg_shipping_cost"]),
                xytext=(6, 6), textcoords="offset points")
ax.set_xlabel("Average lead time (days)")
ax.set_ylabel("Average shipping cost (USD)")
ax.set_title("Shipping Efficiency: Cost vs Lead Time")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "shipping_efficiency.png", dpi=120)
plt.close(fig)

## 8. Monthly sales trend

In [ ]:
monthly = (
    df.groupby("Order Month")
      .agg(total_sales=("Sales", "sum"), total_profit=("Profit", "sum"))
      .reset_index()
      .sort_values("Order Month")
)
monthly["total_sales"] = monthly["total_sales"].round(2)
monthly["total_profit"] = monthly["total_profit"].round(2)

fig, ax = plt.subplots()
ax.plot(monthly["Order Month"], monthly["total_sales"], color=PALETTE[0], label="Sales")
ax.plot(monthly["Order Month"], monthly["total_profit"], color=PALETTE[3], label="Profit")
ax.set_xticks(monthly["Order Month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("USD")
ax.set_title("Monthly Sales and Profit Trend")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "monthly_trend.png", dpi=120)
plt.close(fig)

## 9. Export dashboard data and summary
Every JSON file below is consumed by `docs/dashboard.json`.

In [ ]:
cat.to_json(OUT_DOCS / "sales_profit_by_category.json", orient="records")
reg.to_json(OUT_DOCS / "profit_by_region.json", orient="records")
disc.to_json(OUT_DOCS / "discount_impact.json", orient="records")
ship.to_json(OUT_DOCS / "shipping_efficiency.json", orient="records")
monthly.to_json(OUT_DOCS / "monthly_trend.json", orient="records")

summary = {
    "total_sales": round(float(total_sales), 2),
    "total_profit": round(float(total_profit), 2),
    "profit_margin_pct": round(float(profit_margin * 100), 2),
    "avg_discount_pct": round(float(avg_discount * 100), 2),
    "avg_lead_time_days": round(float(avg_lead_time), 1),
    "shipping_cost_pct_of_sales": round(float(shipping_ratio * 100), 2),
    "orders": int(orders),
    "line_items": int(len(df)),
    "loss_making_subcategories": int(
        (df.groupby("Product Sub-Category")["Profit"].sum() < 0).sum()
    ),
}
with open(ROOT / "outputs" / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## 10. Conclusions
- **Technology** is the highest-margin category; **Office Supplies** drives volume.
- **Discounts above 15%** turn margin negative — cap them.
- **Delivery Truck** is the cheapest ship mode; reserve **Regular Air** for high-priority orders.
- Regional profitability varies widely — rebalance promotions toward profitable regions.